# 05 — Model Training
Train a classifier to categorize light curve signals into:
- **Planet** — real exoplanet transit
- **Eclipsing Binary** — false positive
- **Unknown** — needs more data

We use **Random Forest + XGBoost** on the features extracted in notebook 04.

## 1. Imports

In [ ]:
pip install xgboost

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score)
from sklearn.pipeline import Pipeline
import xgboost as xgb
import pickle

print('Imports OK!')

ModuleNotFoundError: No module named 'xgboost'

## 2. Load Features

In [ ]:
LABELED_FEAT  = '../outputs/labeled_features.csv'
SCIENCE_FEAT  = '../outputs/features.csv'
MODELS_DIR    = '../models/'
os.makedirs(MODELS_DIR, exist_ok=True)

# Load labeled features for training
if os.path.exists(LABELED_FEAT):
    labeled_df = pd.read_csv(LABELED_FEAT)
    labeled_df['tic_id'] = labeled_df['tic_id'].astype(str).str.replace('.0','', regex=False)
    print(f'Labeled samples : {len(labeled_df)}')
    print(labeled_df['label'].value_counts())
else:
    labeled_df = pd.DataFrame()
    print('No labeled features found — will use rule-based classification only')

# Load science features
science_df = pd.read_csv(SCIENCE_FEAT)
science_df['tic_id'] = science_df['tic_id'].astype(str).str.replace('.0','', regex=False)
print(f'\nScience samples : {len(science_df)}')

## 3. Define Feature Columns

In [ ]:
FEATURE_COLS = [
    'period_days',
    'duration_hours',
    'depth_ppm',
    'snr',
    'bls_power',
    'odd_even_diff',
    'secondary_depth',
    'transit_skewness',
    'transit_kurtosis',
    'flux_std',
    'flux_range',
    'outlier_frac',
    'depth_over_std',
    'period_over_dur',
    'in_out_scatter_ratio',
]

# Keep only columns that exist
FEATURE_COLS = [c for c in FEATURE_COLS if c in science_df.columns]
print(f'Using {len(FEATURE_COLS)} features:')
for c in FEATURE_COLS:
    print(f'  {c}')

## 4. Rule-Based Pre-Classification
Even without a trained model, physics gives us clear rules to classify signals.

In [ ]:
def rule_based_classify(row):
    """
    Physics-based classification rules.
    Returns (label, confidence, reason)
    """
    snr           = row.get('snr', 0)
    depth_ppm     = row.get('depth_ppm', 0)
    odd_even_diff = row.get('odd_even_diff', 0)
    sec_depth     = row.get('secondary_depth', 0)
    period        = row.get('period_days', 0)
    duration_h    = row.get('duration_hours', 0)

    # ── Rule 1: Too low SNR = not significant ──
    if snr < 5:
        return 'noise', 0.90, 'SNR too low (<5)'

    # ── Rule 2: Eclipsing Binary indicators ──
    if odd_even_diff > 0.01:
        return 'eclipsing_binary', 0.85, 'Large odd-even depth difference'

    if sec_depth > 0.001:
        return 'eclipsing_binary', 0.80, 'Secondary eclipse detected'

    # ── Rule 3: Depth too deep for planet ──
    if depth_ppm > 100000:   # >10% flux drop = stellar eclipse
        return 'eclipsing_binary', 0.88, 'Transit too deep for planet (>10%)'

    # ── Rule 4: Plausible planet ──
    if snr >= 7 and depth_ppm < 100000 and odd_even_diff < 0.005:
        if 0.5 < period < 13 and 0.5 < duration_h < 16:
            return 'planet_candidate', 0.75, 'Passes all planet criteria'

    # ── Rule 5: Moderate SNR, ambiguous ──
    if 5 <= snr < 7:
        return 'unknown', 0.50, 'Marginal SNR — needs follow-up'

    return 'unknown', 0.40, 'Does not meet any clear criteria'


# Apply to science data
results = science_df.apply(
    lambda row: pd.Series(rule_based_classify(row),
                          index=['rule_label','rule_confidence','rule_reason']),
    axis=1
)
science_df = pd.concat([science_df, results], axis=1)

print('Rule-based classification results:')
print(science_df['rule_label'].value_counts())
print()
print(science_df[['tic_id','snr','depth_ppm','odd_even_diff',
                   'rule_label','rule_confidence','rule_reason']].to_string())

## 5. Train ML Classifier (if labeled data available)

In [ ]:
ml_available = len(labeled_df) >= 4  # Need at least 4 samples

if not ml_available:
    print('⚠️  Not enough labeled data for ML training.')
    print('Using rule-based classification only.')
    print('To enable ML: download more labeled data in notebook 01')
else:
    print(f'Training ML model on {len(labeled_df)} labeled samples...')

    # Prepare data
    labeled_df['tic_id'] = labeled_df['tic_id'].astype(str).str.replace('.0','', regex=False)
    feat_cols_avail = [c for c in FEATURE_COLS if c in labeled_df.columns]

    X = labeled_df[feat_cols_avail].fillna(0).values
    y = labeled_df['label'].values

    le = LabelEncoder()
    y_enc = le.fit_transform(y)

    print(f'Classes : {le.classes_}')
    print(f'X shape : {X.shape}')

    # Train/test split
    if len(X) >= 6:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
        )
    else:
        X_train, X_test = X, X
        y_train, y_test = y_enc, y_enc
        print('⚠️  Too few samples for proper split — using all data for train and test')

    # Random Forest
    rf = Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    RandomForestClassifier(n_estimators=100, random_state=42,
                                          class_weight='balanced'))
    ])
    rf.fit(X_train, y_train)
    rf_score = rf.score(X_test, y_test)
    print(f'\nRandom Forest accuracy : {rf_score:.2%}')

    # XGBoost
    xgb_model = Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    xgb.XGBClassifier(n_estimators=100, random_state=42,
                                     eval_metric='mlogloss',
                                     use_label_encoder=False))
    ])
    xgb_model.fit(X_train, y_train)
    xgb_score = xgb_model.score(X_test, y_test)
    print(f'XGBoost accuracy       : {xgb_score:.2%}')

    # Pick best model
    best_model = rf if rf_score >= xgb_score else xgb_model
    best_name  = 'RandomForest' if rf_score >= xgb_score else 'XGBoost'
    print(f'\nBest model: {best_name}')

    # Save models
    with open(os.path.join(MODELS_DIR, 'random_forest.pkl'), 'wb') as f:
        pickle.dump(rf, f)
    with open(os.path.join(MODELS_DIR, 'xgboost_model.pkl'), 'wb') as f:
        pickle.dump(xgb_model, f)
    with open(os.path.join(MODELS_DIR, 'label_encoder.pkl'), 'wb') as f:
        pickle.dump(le, f)
    with open(os.path.join(MODELS_DIR, 'feature_cols.pkl'), 'wb') as f:
        pickle.dump(feat_cols_avail, f)
    print('Models saved to models/')

## 6. Apply ML Model to Science Data (if trained)

In [ ]:
if ml_available:
    X_sci = science_df[feat_cols_avail].fillna(0).values

    ml_labels = le.inverse_transform(best_model.predict(X_sci))
    ml_probs  = best_model.predict_proba(X_sci).max(axis=1)

    science_df['ml_label']      = ml_labels
    science_df['ml_confidence'] = ml_probs

    print('ML classification results:')
    print(science_df[['tic_id','rule_label','ml_label','ml_confidence']].to_string())
else:
    science_df['ml_label']      = science_df['rule_label']
    science_df['ml_confidence'] = science_df['rule_confidence']
    print('Using rule-based labels as final classification')

## 7. Final Combined Label

In [ ]:
# Use ML label if available, else rule-based
science_df['final_label'] = science_df['ml_label']
science_df['final_confidence'] = science_df['ml_confidence']

# Save final results
science_df.to_csv('../outputs/results.csv', index=False)
print('Final classification results:')
print('=' * 60)
print(science_df[['tic_id','final_label','final_confidence','rule_reason']].to_string())
print('='*60)
print(science_df['final_label'].value_counts())

## 8. Feature Importance Plot (if ML trained)

In [ ]:
if ml_available:
    importances = rf.named_steps['clf'].feature_importances_
    feat_imp_df = pd.DataFrame({
        'feature'    : feat_cols_avail,
        'importance' : importances
    }).sort_values('importance', ascending=True)

    fig, ax = plt.subplots(figsize=(10, 7))
    colors = ['steelblue'] * len(feat_imp_df)
    # Highlight top 3
    top3 = feat_imp_df.nlargest(3, 'importance').index
    for i in top3:
        colors[list(feat_imp_df.index).index(i)] = 'darkorange'

    ax.barh(feat_imp_df['feature'], feat_imp_df['importance'], color=colors[::-1])
    ax.set_xlabel('Feature Importance')
    ax.set_title('Random Forest — Feature Importances\n(orange = top 3)')
    plt.tight_layout()
    plt.savefig('../outputs/plots/feature_importance.png', dpi=150)
    plt.show()
else:
    # Show rule-based decision summary instead
    fig, ax = plt.subplots(figsize=(10, 5))
    label_counts = science_df['final_label'].value_counts()
    colors_map = {
        'planet_candidate' : 'green',
        'eclipsing_binary' : 'red',
        'noise'            : 'gray',
        'unknown'          : 'orange'
    }
    bar_colors = [colors_map.get(l, 'steelblue') for l in label_counts.index]
    ax.bar(label_counts.index, label_counts.values, color=bar_colors, edgecolor='white')
    ax.set_xlabel('Classification')
    ax.set_ylabel('Number of Stars')
    ax.set_title('Rule-Based Classification Summary')
    plt.tight_layout()
    plt.savefig('../outputs/plots/classification_summary.png', dpi=150)
    plt.show()

print('Plot saved!')

## 9. Confusion Matrix (if ML trained)

In [ ]:
if ml_available and len(X_test) > 1:
    y_pred = best_model.predict(X_test)
    cm     = confusion_matrix(y_test, y_pred)
    disp   = ConfusionMatrixDisplay(cm, display_labels=le.classes_)

    fig, ax = plt.subplots(figsize=(7, 6))
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{best_name} Confusion Matrix')
    plt.tight_layout()
    plt.savefig('../outputs/plots/confusion_matrix.png', dpi=150)
    plt.show()

    print('\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=le.classes_))
else:
    print('Confusion matrix requires ML training with enough labeled data.')
    print('Skipping — rule-based results saved in outputs/results.csv')

---
## ✅ Done!
**What was saved:**
- `outputs/results.csv` — final label + confidence for every star
- `models/random_forest.pkl` — trained RF model (if labeled data available)
- `models/xgboost_model.pkl` — trained XGB model
- `outputs/plots/classification_summary.png` — label distribution
- `outputs/plots/feature_importance.png` — which features matter most

**Next Step → Run `06_classification.ipynb`**